# **Stock Market Data Collection**

This notebook collects daily stock market data used in the GameStop analysis. Historical market data are retrieved for GameStop (GME), SPDR S&P 500 ETF Trust (SPY), AMC Entertainment (AMC), and BlackBerry (BB) to support subsequent return analysis, predictive modeling, and event-study analysis.

The collection period extends from October 1, 2020 through March 31, 2021, providing additional pre-event market observations required for the event-study estimation period.

## **1. Setup**

Required Python libraries are imported for market-data retrieval, tabular data processing, and file management.

In [ ]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------

# Standard library
from pathlib import Path

# Data access
import yfinance as yf


## **2. Data Collection Configuration**

The analysis includes four securities: GME as the primary stock of interest, SPY as the broader-market benchmark, and AMC and BB as additional meme-stock securities.

The collection window is defined from October 1, 2020 through March 31, 2021. The broader starting date provides sufficient pre-event observations for estimating expected GME returns in the subsequent event-study analysis.

In [3]:
# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

TICKERS = ["GME", "SPY", "AMC", "BB"]

# Report's methodology table: Oct 1, 2020 - Mar 31, 2021
# (wider than the GME-only window so Event Study has a
# pre-event estimation period to compute "normal" returns against)
START_DATE = "2020-10-01"
END_DATE   = "2021-04-01"

STOCK_DIR = Path("../data/raw/stock")
STOCK_DIR.mkdir(parents=True, exist_ok=True)

## **3. Market Data Collection and Validation**

Historical daily market data are downloaded separately for each selected security. After retrieval, the data structure is standardized and basic validation checks are performed for dataset dimensions, date coverage, missing values, and duplicate trading dates.

Each validated dataset is then saved as a separate raw CSV file for subsequent preprocessing and analysis.

In [4]:
# ---------------------------------------------------------
# Download, flatten, validate, and save each ticker
# ---------------------------------------------------------

stock_data = {}

for ticker in TICKERS:

    print(f"\n=== {ticker} ===")

    df = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=False
    )

    df = df.reset_index()

    # Flatten yfinance MultiIndex columns
    df.columns = [
        col[0] if col[0] == "Date" else col[0]
        for col in df.columns
    ]

    print("Shape:", df.shape)
    print("Date range:", df["Date"].min(), "→", df["Date"].max())
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate dates:", df["Date"].duplicated().sum())

    raw_path = STOCK_DIR / f"{ticker.lower()}_stock.csv"

    df.to_csv(raw_path, index=False)
    print(f"Saved {len(df)} rows to: {raw_path}")

    stock_data[ticker] = df

print("\nAll tickers collected:", list(stock_data.keys()))


=== GME ===
Shape: (125, 7)
Date range: 2020-10-01 00:00:00 → 2021-03-31 00:00:00
Missing values: 0
Duplicate dates: 0
Saved 125 rows to: ../data/raw/stock/gme_stock.csv

=== SPY ===
Shape: (125, 7)
Date range: 2020-10-01 00:00:00 → 2021-03-31 00:00:00
Missing values: 0
Duplicate dates: 0
Saved 125 rows to: ../data/raw/stock/spy_stock.csv

=== AMC ===
Shape: (125, 7)
Date range: 2020-10-01 00:00:00 → 2021-03-31 00:00:00
Missing values: 0
Duplicate dates: 0
Saved 125 rows to: ../data/raw/stock/amc_stock.csv

=== BB ===
Shape: (125, 7)
Date range: 2020-10-01 00:00:00 → 2021-03-31 00:00:00
Missing values: 0
Duplicate dates: 0
Saved 125 rows to: ../data/raw/stock/bb_stock.csv

All tickers collected: ['GME', 'SPY', 'AMC', 'BB']


### **Collection Summary**

Market data were successfully collected for all four securities. Each dataset contains **125 trading-day observations** covering **October 1, 2020 through March 31, 2021**, with **no missing values or duplicate trading dates**.

The resulting raw datasets were saved separately for GME, SPY, AMC, and BB and are subsequently processed in `04_data_preprocessing.ipynb`.